# IWTC Tools: Incremental Rebuild

This notebook provides a fast, deterministic rebuild path for canonical index and graph artifacts for a single world repository.

It replaces full pipeline execution with targeted rebuilds based on simple change flags, writing all outputs directly to the canonical index directory.

### Design Reference
- indexing_system_overview.md

### Repository Descriptor
- world_repository.yml

## Phase P0: Parameters

Define which world repository this notebook operates on and which index version it expects.

This notebook rebuilds index and graph data files. 

**IMPORTANT:** Unlike the bootstraping notebooks, this notebook writes directly to the canonical index directory identified in the descriptor file under indexes.path.

In [15]:
# Phase P0: Parameters
LAST_PHASE_RUN = "P0"

# Absolute path to the world_repository.yml descriptor.
WORLD_REPOSITORY_DESCRIPTOR = (
    "/Users/charissophia/obsidian/Iron Wolf Trading Company/_meta/descriptors/world_repository.yml"
)

# Index version to load (must match previously generated artifacts)
INDEX_VERSION = "V0"
INDEX_VERSION_SUFFIX = INDEX_VERSION.lower()

# Indicate which inputs have changed
CHANGED_SOURCES        = False
CHANGED_ENTITIES       = False
CHANGED_ALIASES        = False
CHANGED_PREDICATES     = False
CHANGED_RELATIONSHIPS  = False

# Optional: Force Rebuild
FORCE_REBUILD_SOURCES  = False
FORCE_REBUILD_EVIDENCE = False
FORCE_REBUILD_SEMANTIC = False

# Internal run metadata (do not edit)
from datetime import datetime
print(f"Notebook run initialized at: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
del datetime

print("\nChanged inputs:")
print(f"{'Sources:':16} {CHANGED_SOURCES}")
print(f"{'Entities:':16} {CHANGED_ENTITIES}")
print(f"{'Aliases:':16} {CHANGED_ALIASES}")
print(f"{'Predicates:':16} {CHANGED_PREDICATES}")
print(f"{'Relationships:':16} {CHANGED_RELATIONSHIPS}")

print("\nForce rebuild:")
print(f"{'Raw sources:':16} {FORCE_REBUILD_SOURCES}")
print(f"{'Evidence graph:':16} {FORCE_REBUILD_EVIDENCE}")
print(f"{'Semantic graph:':16} {FORCE_REBUILD_SEMANTIC}")

Notebook run initialized at: 2026-03-25 19:53

Changed inputs:
Sources:         False
Entities:        False
Aliases:         False
Predicates:      False
Relationships:   False

Force rebuild:
Raw sources:     False
Evidence graph:  False
Semantic graph:  False


## Phase P1: Descriptor Load and Path Resolution

Load the world repository descriptor and resolve all required paths.

Fails fast if required fields are missing.

In [16]:
# Phase P1: Descriptor Load + Path Resolution
LAST_PHASE_RUN = "P1"

from pathlib import Path
import yaml
from IPython.display import display

p1_errors = []


# Load descriptor
descriptor_path = Path(WORLD_REPOSITORY_DESCRIPTOR)

if descriptor_path.exists():
    with descriptor_path.open("r", encoding="utf-8") as f:
        descriptor = yaml.safe_load(f)
    del f

    if not isinstance(descriptor, dict):
        raise ValueError("Descriptor file did not load as a dictionary.")
else:
    raise FileNotFoundError(f"Descriptor not found: {WORLD_REPOSITORY_DESCRIPTOR}")

del descriptor_path


# Resolve world root
WORLD_ROOT = descriptor.get("world_root")

if WORLD_ROOT and Path(WORLD_ROOT).is_absolute():
    WORLD_ROOT = Path(WORLD_ROOT)
else:
    raise ValueError("Descriptor must define world_root as an absolute path.")


# Resolve indexes path
INDEXES_PATH = None
INDEXES_RELPATH = descriptor.get("indexes", {}).get("path")

if INDEXES_RELPATH:
    indexes_path = Path(INDEXES_RELPATH)

    if indexes_path.is_absolute():
        INDEXES_PATH = indexes_path
    else:
        INDEXES_PATH = WORLD_ROOT / indexes_path

    if not INDEXES_PATH.exists():
        p1_errors.append(f"Indexes path not found: {INDEXES_RELPATH}")

    del indexes_path
else:
    p1_errors.append("Descriptor missing indexes.path")


# Resolve vocabulary paths
VOCAB_ENTITIES_PATH = None
VOCAB_ENTITIES_RELPATH = None
VOCAB_ALIASES_PATH = None
VOCAB_ALIASES_RELPATH = None
VOCAB_RELATIONSHIPS_PATH = None
VOCAB_RELATIONSHIPS_RELPATH = None
VOCAB_PREDICATES_PATH = None
VOCAB_PREDICATES_RELPATH = None

def resolve_vocab_path(key):
    relpath = descriptor.get("vocabulary", {}).get(key)

    if relpath:
        path_obj = Path(relpath)

        if path_obj.is_absolute():
            abspath = path_obj
        else:
            abspath = WORLD_ROOT / path_obj

        if not abspath.exists():
            p1_errors.append(f"Vocabulary file not found: {relpath}")

        del path_obj
        return abspath, relpath
    else:
        p1_errors.append(f"Descriptor missing vocabulary.{key}")
        return None, None

VOCAB_ENTITIES_PATH, VOCAB_ENTITIES_RELPATH = resolve_vocab_path("entities")
VOCAB_ALIASES_PATH, VOCAB_ALIASES_RELPATH = resolve_vocab_path("aliases")
VOCAB_RELATIONSHIPS_PATH, VOCAB_RELATIONSHIPS_RELPATH = resolve_vocab_path("relationships")
VOCAB_PREDICATES_PATH, VOCAB_PREDICATES_RELPATH = resolve_vocab_path("predicates")


# Resolve source paths (no loading yet)
SOURCE_PATHS = []
read_paths = descriptor.get("sources", {}).get("read_paths", [])

if isinstance(read_paths, list) and len(read_paths) > 0:
    for entry in read_paths:
        if isinstance(entry, dict):
            relpath = entry.get("path")
            source_type = entry.get("type")
        else:
            relpath = entry
            source_type = None

        if relpath:
            path_obj = Path(relpath)

            if path_obj.is_absolute():
                abspath = path_obj
            else:
                abspath = WORLD_ROOT / path_obj

            SOURCE_PATHS.append({
                "path": abspath,
                "relpath": relpath,
                "type": source_type,
            })

            if not abspath.exists():
                p1_errors.append(f"Source path not found: {relpath}")

            del path_obj, abspath
        else:
            p1_errors.append("A sources.read_paths entry is missing its path value.")

        del entry, relpath, source_type
else:
    p1_errors.append("Descriptor missing sources.read_paths or it is empty.")

if isinstance(read_paths, list) and len(read_paths) > 0 and len(SOURCE_PATHS) == 0:
    p1_errors.append("No valid source paths were resolved from sources.read_paths.")

del read_paths


# Raise once, with complete list
if p1_errors:
    print("Phase P1 failed. Please verify the descriptor file.\n")
    for msg in p1_errors:
        print(f"- {msg}")
    raise ValueError("Phase P1 failed. See error list above.")

del p1_errors, descriptor, yaml


# Summary
print("World Root:   ", WORLD_ROOT)
print("Indexes:      ", INDEXES_RELPATH)
print("Entities:     ", VOCAB_ENTITIES_RELPATH)
print("Aliases:      ", VOCAB_ALIASES_RELPATH)
print("Predicates:   ", VOCAB_PREDICATES_RELPATH)
print("Relationships:", VOCAB_RELATIONSHIPS_RELPATH)

print("\nSources:", len(SOURCE_PATHS), "read paths resolved")
display([s["relpath"] for s in SOURCE_PATHS])

print("\nPhase P1 OK: descriptor loaded and paths resolved.")

World Root:    /Users/charissophia/obsidian/Iron Wolf Trading Company
Indexes:       _meta/indexes
Entities:      _meta/indexes/vocab_entities.csv
Aliases:       _meta/indexes/vocab_aliases.csv
Predicates:    _meta/indexes/vocab_predicates.csv
Relationships: _meta/indexes/world_relationships.csv

Sources: 5 read paths resolved


['_local/auto_transcripts',
 '_local/pbp_transcripts',
 '_local/session_notes',
 '_local/planning_notes',
 '_local/recollections']


Phase P1 OK: descriptor loaded and paths resolved.


## Phase P2: Load Canonical Vocabulary Tables

Load the canonical vocabulary and semantic input tables from the index directory.

These files are small and may be reused across multiple later phases, so they are loaded once here and retained in memory.

Because these files are human-generated, column names are normalized here to a stable internal schema for later phases.

In [ ]:
# Phase P2: Load and Normalize Canonical Vocabulary Tables
LAST_PHASE_RUN = "P2"

import pandas as pd
import re

p2_errors = []


# Allowed column variants
ENTITY_COLS = {
    "entity_id": ["entity_id", "id"],
    "canonical": ["canonical", "canonical_name", "name"],
}

ALIAS_COLS = {
    "entity_id": ["entity_id", "id"],
    "alias": ["alias", "alt", "alternate"],
}

PREDICATE_COLS = {
    "predicate": ["predicate"],
    "reverse_predicate": ["reverse_predicate", "reverse"],
    "include": ["include"],
    "relationship_class": ["relationship_class", "class"],
    "priority": ["priority", "weight", "score"],
    "cost": ["cost"],
}

RELATIONSHIP_COLS = {
    "subject_id": ["subject_id", "subject", "from_id"],
    "predicate": ["predicate", "relationship", "relation", "edge", "verb"],
    "object_id": ["object_id", "object", "target_id", "to_id"],
}


def load_and_normalize_csv(path_obj, col_map, key_label):
    if path_obj.exists():
        raw_df = pd.read_csv(path_obj, dtype=str).fillna("")
    else:
        p2_errors.append(f"{key_label} file not found: {path_obj}")
        return None

    rename = {}
    for semantic, options in col_map.items():
        found = next((c for c in options if c in raw_df.columns), None)
        if found:
            rename[found] = semantic

    out = raw_df.rename(columns=rename)
    missing = [k for k in col_map.keys() if k not in out.columns]

    if len(missing) == 0:
        return out[list(col_map.keys())]

    p2_errors.append(f"[{key_label}] Missing required columns: {missing}")
    return None


# Load and normalize canonical vocab tables
DF_VOCAB_ENTITIES = load_and_normalize_csv(
    VOCAB_ENTITIES_PATH,
    ENTITY_COLS,
    "Entities",
)

DF_VOCAB_ALIASES = load_and_normalize_csv(
    VOCAB_ALIASES_PATH,
    ALIAS_COLS,
    "Aliases",
)

DF_VOCAB_PREDICATES = load_and_normalize_csv(
    VOCAB_PREDICATES_PATH,
    PREDICATE_COLS,
    "Predicates",
)

DF_VOCAB_RELATIONSHIPS = load_and_normalize_csv(
    VOCAB_RELATIONSHIPS_PATH,
    RELATIONSHIP_COLS,
    "Relationships",
)


# Normalize predicate data types
if DF_VOCAB_PREDICATES is not None:
    DF_VOCAB_PREDICATES["include"] = (
        DF_VOCAB_PREDICATES["include"]
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )

    DF_VOCAB_PREDICATES["priority"] = pd.to_numeric(
        DF_VOCAB_PREDICATES["priority"],
        errors="coerce",
    )

    DF_VOCAB_PREDICATES["cost"] = pd.to_numeric(
        DF_VOCAB_PREDICATES["cost"],
        errors="coerce",
    )


# Build consolidated linking vocabulary (canonicals + aliases)
lookup_rows = []

for _, r in DF_VOCAB_ENTITIES.iterrows():
    if r.get("entity_id", "") and r.get("canonical", ""):
        lookup_rows.append(
            {
                "vocab": r["canonical"],
                "vocab_id": r["entity_id"],
                "canonical": r["canonical"],
                "vocab_kind": "canonical",
            }
        )
canon_by_id = dict(zip(DF_VOCAB_ENTITIES["entity_id"], DF_VOCAB_ENTITIES["canonical"]))
for _, r in DF_VOCAB_ALIASES.iterrows():
    if r.get("entity_id", "") and r.get("alias", ""):
        lookup_rows.append(
            {
                "vocab": r["alias"],
                "vocab_id": r["entity_id"],
                "canonical": canon_by_id.get(r["entity_id"], ""),
                "vocab_kind": "alias",
            }
        )
del canon_by_id, r

DF_VOCAB_LOOKUP = pd.DataFrame(lookup_rows).drop_duplicates(subset=["vocab", "vocab_id"]).reset_index(drop=True)
DF_VOCAB_LOOKUP["vocab_len"] = DF_VOCAB_LOOKUP["vocab"].str.len()
DF_VOCAB_LOOKUP = DF_VOCAB_LOOKUP.sort_values(["vocab_len", "vocab"], ascending=[False, True]).reset_index(drop=True)
DF_VOCAB_LOOKUP["vocab_norm"] = DF_VOCAB_LOOKUP["vocab"].astype(str).str.strip().str.lower()

# Compile regex patterns (boundary-aware). Avoid backslashes inside f-string expressions.
patterns = []
for vocab in DF_VOCAB_LOOKUP["vocab"]:
    esc = re.escape(vocab)
    esc_ws = esc.replace(r"\ ", r"\s+")
    patterns.append(re.compile(rf"(?<!\w){esc_ws}(?!\w)", re.IGNORECASE))
    del esc, esc_ws
DF_VOCAB_LOOKUP["pattern"] = patterns
del vocab


# Normalize semantic relationsihps
DF_RELATIONSHIP_SEMANTICS = (
    DF_VOCAB_RELATIONSHIPS
    .assign(
        subject_type=lambda df: df["subject_id"].str.split("_", n=1).str[0],
        object_type=lambda df: df["object_id"].str.split("_", n=1).str[0],
    )
    .merge(
        DF_VOCAB_PREDICATES,
        on="predicate",
        how="left",
    )
    .assign(
        pair_type=lambda df: df["subject_type"] + "|" + df["object_type"],
        symmetric=lambda df: (
            df["predicate"].str.strip().str.lower()
            == df["reverse_predicate"].str.strip().str.lower()
        ),
    )
    [
        [
            "subject_id",
            "subject_type",
            "predicate",
            "object_id",
            "object_type",
            "pair_type",
            "include",
            "symmetric",
            "relationship_class",
        ]
    ]
    .sort_values(
        ["predicate", "subject_id", "object_id"],
        ascending=[True, True, True],
    )
    .reset_index(drop=True)
)



# Raise once, with complete list
if p2_errors:
    print("Phase P2 failed. Please verify the vocabulary files.\n")
    for msg in p2_errors:
        print(f"- {msg}")
    raise ValueError("Phase P2 failed. See error list above.")


# Summary
print(f"{'Entities:':14} {VOCAB_ENTITIES_RELPATH:40} ({len(DF_VOCAB_ENTITIES)} rows)")
print(f"{'Aliases:':14} {VOCAB_ALIASES_RELPATH:40} ({len(DF_VOCAB_ALIASES)} rows)")
print(f"{'Link vocabs:':14} {'(in memory)':40} ({len(DF_VOCAB_LOOKUP)} rows)")
print(f"{'Predicates:':14} {VOCAB_PREDICATES_RELPATH:40} ({len(DF_VOCAB_PREDICATES)} rows)")
print(f"{'Relationships:':14} {VOCAB_RELATIONSHIPS_RELPATH:40} ({len(DF_VOCAB_RELATIONSHIPS)} rows)")
print("\nNormalized semantic relationships:")
print(f"- Input relationships: {len(DF_VOCAB_RELATIONSHIPS):>8}")
print(f"- Output rows:         {len(DF_RELATIONSHIP_SEMANTICS):>8}")
print(f"- Distinct predicates: {DF_RELATIONSHIP_SEMANTICS['predicate'].nunique():>8}")
print(f"- Distinct pair types: {DF_RELATIONSHIP_SEMANTICS['pair_type'].nunique():>8}")
print()


display(DF_VOCAB_ENTITIES.head(3))
display(DF_VOCAB_ALIASES.head(3))
display(DF_VOCAB_LOOKUP.head(3))
display(DF_VOCAB_PREDICATES.head(3))
display(DF_VOCAB_RELATIONSHIPS.head(3))
display(DF_RELATIONSHIP_SEMANTICS.head(3))

print("\nPhase P2 OK: canonical vocabulary tables loaded and normalized.")


# Clean up variables we won't need ongoing
del p2_errors, lookup_rows, patterns
del ALIAS_COLS, ENTITY_COLS, PREDICATE_COLS, RELATIONSHIP_COLS
del VOCAB_ENTITIES_PATH, VOCAB_ENTITIES_RELPATH
del VOCAB_ALIASES_PATH, VOCAB_ALIASES_RELPATH
del VOCAB_PREDICATES_PATH, VOCAB_PREDICATES_RELPATH
del VOCAB_RELATIONSHIPS_PATH, VOCAB_RELATIONSHIPS_RELPATH

In [ ]:
# ----------------------
# Validate canonical relationship against entity vocabulary
# ----------------------
KNOWN_ENTITY_IDS = set(DF_VOCAB_ENTITIES["entity_id"])

unknown_subjects = (
    DF_VOCAB_RELATIONSHIPS.loc[~DF_VOCAB_RELATIONSHIPS["subject_id"].isin(KNOWN_ENTITY_IDS), "subject_id"]
    .drop_duplicates()
    .sort_values()
)

unknown_objects = (
    DF_VOCAB_RELATIONSHIPS.loc[~DF_VOCAB_RELATIONSHIPS["object_id"].isin(KNOWN_ENTITY_IDS), "object_id"]
    .drop_duplicates()
    .sort_values()
)

if len(unknown_subjects) or len(unknown_objects):

    print("Unknown subject_ids:")
    display(unknown_subjects)

    print("\nUnknown object_ids:")
    display(unknown_objects)

    raise ValueError(
        "Canonical relationships reference unknown entity IDs.\n\n"
    )

print("\nAll relationship endpoints reference known entities.")
print(f"- Relationships checked: {len(DF_VOCAB_RELATIONSHIPS)}")
print(f"- Distinct predicates:   {DF_VOCAB_RELATIONSHIPS['predicate'].nunique()}")

print("\nPredicate distribution:")
display(
    DF_VOCAB_RELATIONSHIPS["predicate"]
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="relationship_count")
)

del KNOWN_ENTITY_IDS, unknown_subjects, unknown_objects

## Phase P3: Rebuild Source-Derived Indexes

Rebuild the source-derived entity index files if source content or source-matching vocabulary has changed.

Note that only .md and .txt files are currently loaded.

This phase writes directly to the canonical index directory.

In [ ]:
# Phase P3: Rebuild Source-Derived Indexes
LAST_PHASE_RUN = "P3"

CHANGED_SOURCE_INDEXES = False

if (
    FORCE_REBUILD_SOURCES
    or CHANGED_SOURCES
    or CHANGED_ENTITIES
    or CHANGED_ALIASES
):
    print("Phase P3 will run:")
    print(f"{'  Force rebuild sources:':24} {FORCE_REBUILD_SOURCES}")
    print(f"{'  Changed sources:':24} {CHANGED_SOURCES}")
    print(f"{'  Changed entities:':24} {CHANGED_ENTITIES}")
    print(f"{'  Changed aliases:':24} {CHANGED_ALIASES}")

    # ----------------------
    # Discover source files
    # ----------------------
    SOURCE_FILES = []

    for source in SOURCE_PATHS:
        if source["path"].exists() and source["path"].is_dir():
            for p in source["path"].rglob("*"):
                if p.is_file() and p.suffix.lower() in {".md", ".txt"}:
                    SOURCE_FILES.append({
                        "path": p,
                        "relpath": str(p.relative_to(WORLD_ROOT)),
                        "source_type": source["type"],
                    })
        else:
            if source["path"].is_file() and source["path"].suffix.lower() in {".md", ".txt"}:
                SOURCE_FILES.append({
                    "path": source["path"],
                    "relpath": str(source["path"].relative_to(WORLD_ROOT)),
                    "source_type": source["type"],
                })

    print(f"\nSource files discovered: {len(SOURCE_FILES)}")
    display([s["relpath"] for s in SOURCE_FILES[:10]])
    if len(SOURCE_FILES) > 10:
        print("  ... (truncated)")

    print("\nBy source type:")
    print(
        "  "
        + pd.Series([s["source_type"] or "untyped" for s in SOURCE_FILES])
            .value_counts().to_string().replace("\n", "\n  ")
    )
    del p, source

    # ----------------------
    # Load source files into memory as raw lines
    # ----------------------
    LOADED_SOURCES = []
    
    for item in SOURCE_FILES:
        suffix = item["path"].suffix.lower()
    
        if suffix in (".md", ".txt"):
            text = item["path"].read_text(encoding="utf-8", errors="replace")
            lines = text.splitlines()
            del text
    
            LOADED_SOURCES.append({
                "path": item["path"],
                "relpath": item["relpath"],
                "source_type": item["source_type"],
                "lines": lines,
            })
        else:
            raise ValueError(f"Unsupported file type: {item['path']}")
    
    
    # Summary
    print(f"\nLoaded sources: {len(LOADED_SOURCES)}")
    
    for s in LOADED_SOURCES[:10]:
        print(f"  {s['relpath']}  ({len(s['lines'])} lines)")
    
    if len(LOADED_SOURCES) > 10:
        print("  ... (truncated)")
        
    # Clean up
    del SOURCE_FILES, item, suffix, s
    
    # ----------------------
    # Consolidated header-driven chunking
    # ----------------------
    TIME_LIKE_REGEX = r"\d{1,2}:\d{2}(?::\d{2})?(?:\s*[AP]M)?"
    
    # Header regexes are evaluated in order; first match wins.
    HEADER_REGEXES = [
        ("auto_ts"   , re.compile(r"^\s*\d{1,2}:\d{2}(?::\d{2})?\s*$")),
        ("pbp_hash"  , re.compile(rf"^\s*(?:\d+\.\s*)?(?:[*-]\s*)?###\s+.*{TIME_LIKE_REGEX}.*$")),
        ("pbp_forum" , re.compile(rf"^\s*>?\s*\*\*.*{TIME_LIKE_REGEX}.*\*\*\s*$")),
        ("session"   , re.compile(r"^\s*(?:\d+\.\s*)?(?:[>#*_\-\s]+)?session\s+(?!notes\b)\S.*$", re.IGNORECASE)),
        ("md_heading", re.compile(r"^\s*(?:[*\-]\s*)?#{1,6}\s+\S.*$")),
    ]
    
    CHUNKS_V0 = []
    chunk_global_id = 1
    
    for src in LOADED_SOURCES:
        current_kind = "preamble"
        current_lines = []
        chunk_start_line = 1
    
        for idx, line in enumerate(src["lines"], start=1):
            matched_kind = next(
                (kind for kind, header_regex in HEADER_REGEXES if header_regex.match(line)),
                None,
            )
    
            if matched_kind:
                if current_lines:
                    CHUNKS_V0.append(
                        {
                            "chunk_id": chunk_global_id,
                            "source_type": src["source_type"],
                            "relpath": src["relpath"],
                            "start_line": chunk_start_line,
                            "end_line": idx - 1,
                            "header_kind": current_kind,
                            "lines": list(current_lines),
                        }
                    )
                    chunk_global_id += 1
    
                current_kind = matched_kind
                current_lines = [line]
                chunk_start_line = idx
            else:
                current_lines.append(line)
            
        if current_lines:
            CHUNKS_V0.append(
                {
                    "chunk_id": chunk_global_id,
                    "source_type": src["source_type"],
                    "relpath": src["relpath"],
                    "start_line": chunk_start_line,
                    "end_line": len(src["lines"]),
                    "header_kind": current_kind,
                    "lines": list(current_lines),
                }
            )
            chunk_global_id += 1
    
    
    print(f"\nChunks created: {len(CHUNKS_V0)} from {len(LOADED_SOURCES)} files.")
    
    print("\nBy header kind:")
    print(
        "  "
        + pd.Series([c["header_kind"] for c in CHUNKS_V0])
            .value_counts()
            .to_string()
            .replace("\n", "\n  ")
    )
    print()
    display(
        pd.DataFrame(CHUNKS_V0)[
            ["chunk_id", "relpath", "start_line", "end_line", "header_kind"]
        ].head(10)
    )
    
    del TIME_LIKE_REGEX, HEADER_REGEXES, chunk_global_id
    del src, current_kind, current_lines, chunk_start_line, idx, line, matched_kind 

    # ----------------------
    # Link entities to chunks
    # ----------------------
    from datetime import datetime
    print(f"Start entity to chunk linking (expect 2-4 minutes runtime): {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    del datetime
    
    EXCLUDE_SOURCE_TYPES = {"auto_transcripts"}
    MAX_SNIPPET_CHARS = 220

    vocabs = DF_VOCAB_LOOKUP["vocab"].tolist()
    entity_ids = DF_VOCAB_LOOKUP["vocab_id"].tolist()
    canonicals = DF_VOCAB_LOOKUP["canonical"].tolist()
    match_kinds = DF_VOCAB_LOOKUP["vocab_kind"].tolist()
    patterns = DF_VOCAB_LOOKUP["pattern"].tolist()

    rows = []
    mention_id = 1

    for chunk in CHUNKS_V0:
        source_type = chunk.get("source_type", "unknown")

        if source_type not in EXCLUDE_SOURCE_TYPES:
            lines = chunk.get("lines", [])
            header_kind = chunk.get("header_kind")

            # Drop pbp/session header line (metadata, not narrative)
            if header_kind in {"pbp_hash", "pbp_forum", "session"} and lines:
                content_lines = lines[1:]
                content_start_line = (chunk.get("start_line") or 1) + 1
            else:
                content_lines = lines
                content_start_line = chunk.get("start_line") or 1

            concat_text = " ".join(" ".join(content_lines).split())

            if concat_text:
                if len(concat_text) > MAX_SNIPPET_CHARS:
                    snippet = concat_text[: MAX_SNIPPET_CHARS - 3] + "..."
                else:
                    snippet = concat_text

                # Longest-first matching with span masking
                consumed = [False] * len(concat_text)

                for vocab, entity_id, canonical, match_kind, pat in zip(
                    vocabs, entity_ids, canonicals, match_kinds, patterns
                ):
                    hits = [(m.start(), m.end()) for m in pat.finditer(concat_text)]

                    if hits:
                        kept = []

                        for a, b in hits:
                            if a >= 0 and b > a:
                                if not any(consumed[a:b]):
                                    kept.append((a, b))

                        if kept:
                            for a, b in kept:
                                for k in range(a, b):
                                    consumed[k] = True
                                del k

                            rows.append(
                                {
                                    "mention_id": mention_id,
                                    "entity_id": entity_id,
                                    "canonical": canonical,
                                    "matched_vocab": vocab,
                                    "match_kind": match_kind,
                                    "match_count_in_chunk": len(kept),

                                    "chunk_id": chunk["chunk_id"],
                                    "source_type": chunk["source_type"],
                                    "relpath": chunk["relpath"],
                                    "chunk_start_line": chunk["start_line"],
                                    "chunk_end_line": chunk["end_line"],
                                    "header_kind": chunk["header_kind"],

                                    "content_start_line": content_start_line,
                                    "snippet": snippet,
                                }
                            )
                            mention_id += 1
                            
    ENTITY_MENTIONS_V0 = pd.DataFrame(rows)

    print(
        f"\nChunks scanned for entity linking: "
        f"{sum(1 for c in CHUNKS_V0 if c.get('source_type') not in EXCLUDE_SOURCE_TYPES)}"
    )
    print(f"Entity mentions: {len(ENTITY_MENTIONS_V0)} rows")

    if len(ENTITY_MENTIONS_V0) > 0:
        display(ENTITY_MENTIONS_V0.head(10))

    del EXCLUDE_SOURCE_TYPES, MAX_SNIPPET_CHARS
    del chunk, rows, mention_id, lines, source_type
    del vocabs, entity_ids, canonicals, match_kinds, patterns

    # ----------------------
    # Build indexes in memory
    # ----------------------
    INDEX_SOURCE_FILES_V0 = (
        pd.DataFrame(CHUNKS_V0)[["relpath", "source_type"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    
    def _uniq_sorted(series):
        vals = [x for x in series.dropna().astype(str).tolist() if x != ""]
        return sorted(set([x for x in series.dropna().astype(str).tolist() if x != ""]))
    
    INDEX_CHUNK_TO_ENTITIES_V0 = (
        ENTITY_MENTIONS_V0.groupby(
            ["chunk_id", "source_type", "relpath", "chunk_start_line", "chunk_end_line"],
            dropna=False
        )
        .agg(
            entity_ids=("entity_id", lambda s: "|".join(_uniq_sorted(s))),
            canonicals=("canonical", lambda s: "|".join(_uniq_sorted(s))),
            entity_count=("entity_id", lambda s: len(set([x for x in s.tolist() if pd.notna(x) and str(x) != ""]))),
            matched_vocabs=("matched_vocab", lambda s: "|".join(_uniq_sorted(s))),
            match_kinds=("match_kind", lambda s: "|".join(_uniq_sorted(s))),
        )
        .reset_index()
        .sort_values(["chunk_id"], ascending=[True])
        .reset_index(drop=True)
    )

    INDEX_ENTITY_TO_CHUNKS_V0 = (
        ENTITY_MENTIONS_V0.groupby(["entity_id", "canonical"], dropna=False)
          .agg(
              chunk_ids=("chunk_id", lambda s: "|".join([str(x) for x in _uniq_sorted(s)])),
              chunk_count=("chunk_id", lambda s: len(set(s.tolist()))),
              file_relpaths=("relpath", lambda s: "|".join(_uniq_sorted(s))),
              file_count=("relpath", lambda s: len(set([x for x in s.tolist() if pd.notna(x) and str(x) != ""]))),
          )
          .reset_index()
          .sort_values(["chunk_count", "file_count", "entity_id"], ascending=[False, False, True])
          .reset_index(drop=True)
    )

    del a,b, entity_id, header_kind, kept, hits
    del content_lines, content_start_line, concat_text, snippet
    del consumed, vocab, canonical, match_kind, pat


    # ----------------------
    # Write indexes out to files
    # ----------------------
    INDEX_ENTITY_TO_CHUNKS_V0.to_csv(INDEXES_PATH / "index_entity_to_chunks_v0.csv", index=False, encoding="utf-8")
    INDEX_CHUNK_TO_ENTITIES_V0.to_csv(INDEXES_PATH / "index_chunk_to_entities_v0.csv", index=False, encoding="utf-8")
    INDEX_SOURCE_FILES_V0.to_csv(INDEXES_PATH / "index_source_files_v0.csv", index=False, encoding="utf-8")
    
    print(f" - {INDEXES_PATH}/index_entity_to_chunks_v0.csv")
    print(f" - {INDEXES_PATH}/index_chunk_to_entities_v0.csv")
    print(f" - {INDEXES_PATH}/index_source_files_v0.csv")
    
    display(INDEX_ENTITY_TO_CHUNKS_V0.head(3))
    display(INDEX_CHUNK_TO_ENTITIES_V0.head(3))
    display(INDEX_SOURCE_FILES_V0.head(3))

    CHANGED_SOURCE_INDEXES = True

# ----------------------
# resume overall structure here
# ----------------------
else:
    print("Phase P3 skipped. No inputs changed.")

## Phase P4: Rebuild Evidence Graphs

In [ ]:
# Phase P4: Rebuild Evidence Graphs
LAST_PHASE_RUN = "P4"
CHANGED_EVIDENCE_GRAPHS = False

if (
    FORCE_REBUILD_EVIDENCE
    or CHANGED_SOURCE_INDEXES
    or CHANGED_ENTITIES
    or CHANGED_ALIASES
):
    print("Phase P4 will run:")
    print(f"{'  Force rebuild evidence:':24} {FORCE_REBUILD_EVIDENCE}")
    print(f"{'  Changed indexes:':24} {CHANGED_SOURCE_INDEXES}")
    print(f"{'  Changed entities:':24} {CHANGED_ENTITIES}")
    print(f"{'  Changed aliases:':24} {CHANGED_ALIASES}")

    # ----------------------
    # Build nodes
    # ----------------------
    nodes = []

    # Entity nodes
    nodes.append(
        DF_VOCAB_ENTITIES.assign(
            node_id=lambda d: d["entity_id"].astype(str),
            node_type=lambda d: d["entity_id"].astype(str).str.split("_", n=1).str[0],
            label=lambda d: d["canonical"].astype(str),
        ).loc[:, ["node_id", "node_type", "label"]]
    )

    # Vocab nodes
    nodes.append(
        DF_VOCAB_LOOKUP.assign(
            node_id=lambda d: "vocab:" + d["vocab_norm"].astype(str),
            node_type="vocab",
            label=lambda d: d["vocab"].astype(str),
        ).loc[:, ["node_id", "node_type", "label"]]
    )

    # Chunk nodes
    nodes.append(
        INDEX_CHUNK_TO_ENTITIES_V0.assign(
            node_id=lambda d: "chunk_" + d["chunk_id"].astype(int).astype(str),
            node_type="chunk",
            label=lambda d: "chunk_" + d["chunk_id"].astype(int).astype(str),
        ).loc[:, ["node_id", "node_type", "label"]]
    )

    # File nodes
    nodes.append(
        INDEX_SOURCE_FILES_V0.assign(
            node_id=lambda d: "file:" + d["relpath"].astype(str),
            node_type=lambda d: d["source_type"].astype(str),
            label=lambda d: d["relpath"].astype(str),
        ).loc[:, ["node_id", "node_type", "label"]]
    )

    # Create the graph
    GRAPH_EVIDENCE_NODES_V0 = (
        pd.concat(nodes, ignore_index=True)
          .drop_duplicates(subset=["node_id"])
          .sort_values(["node_type", "node_id"])
          .reset_index(drop=True)
    )
    del nodes

    # output
    print(f"\nEvidence nodes built: {len(GRAPH_EVIDENCE_NODES_V0)}")
    
    print("\nCounts by node_type:")
    display(GRAPH_EVIDENCE_NODES_V0["node_type"].value_counts().to_frame("count"))
    
    print("\nSample evidence nodes:")
    display(GRAPH_EVIDENCE_NODES_V0.sample(min(5, len(GRAPH_EVIDENCE_NODES_V0)), random_state=7))

    # ----------------------
    # Build edges
    # ----------------------
    edge_rows = []

    # contains, mentions, cooccurs_with
    for _, r in INDEX_CHUNK_TO_ENTITIES_V0.loc[:, ["chunk_id", "relpath", "matched_vocabs", "entity_ids"]].iterrows():
        chunk_node = f"chunk_{int(r['chunk_id'])}"
        file_node = f"file:{r['relpath']}"
    
        # file contains chunk
        edge_rows.append((file_node, "contains", chunk_node, pd.NA))
    
        # chunk mentions vocab (pipe-delimited)
        for v in (x.strip() for x in str(r["matched_vocabs"]).split("|")):
            if v:
                edge_rows.append((chunk_node, "mentions", f"vocab:{v.lower()}", pd.NA))
    
        entity_ids = sorted({e.strip() for e in str(r["entity_ids"]).split("|") if e.strip()})
        for i in range(len(entity_ids)):
            for j in range(i + 1, len(entity_ids)):
                edge_rows.append((entity_ids[i], "cooccurs_with", entity_ids[j], 1))

    # refers_to
    for _, r in DF_VOCAB_LOOKUP.loc[:, ["vocab_norm", "vocab_id"]].iterrows():

        # Translate vocab_norm ("shadowboy") -> graph node id ("vocab:shadowboy")
        subject = f"vocab:{str(r['vocab_norm']).strip()}"
    
        # vocab_id is already the target node id (entity_id or player_entity_id)
        object_ = str(r["vocab_id"]).strip()
    
        edge_rows.append((subject, "refers_to", object_, pd.NA))

    # Create the graph
    GRAPH_EVIDENCE_EDGES_V0 = (
        pd.DataFrame(edge_rows, columns=["subject", "predicate", "object", "weight"])
          .groupby(["subject", "predicate", "object"], as_index=False)
          .agg(weight=("weight", lambda s: s.sum(min_count=1)))
          .sort_values(["predicate", "subject", "object"], ascending=[True, True, True])
          .reset_index(drop=True)
    )

    del edge_rows, r, chunk_node, file_node, v, entity_ids, i, j, subject, object_

    # Output
    print(f"Evidence edges built: {len(GRAPH_EVIDENCE_EDGES_V0)}\n")
    print("Evidence edge counts by predicate:\n")
    
    display(
        GRAPH_EVIDENCE_EDGES_V0
            .groupby("predicate", as_index=False)
            .size()
            .rename(columns={"size": "edge_count"})
            .sort_values("edge_count", ascending=False)
            .reset_index(drop=True)
    )
    
    display(GRAPH_EVIDENCE_EDGES_V0.sample(3))

    # ----------------------
    # Write indexes out to files
    # ----------------------
    GRAPH_EVIDENCE_NODES_V0.to_csv(INDEXES_PATH / "graph_evidence_nodes_v0.csv", index=False, encoding="utf-8")
    GRAPH_EVIDENCE_EDGES_V0.to_csv(INDEXES_PATH / "graph_evidence_edges_v0.csv", index=False, encoding="utf-8")
    
    print(f" - {INDEXES_PATH}/graph_evidence_nodes_v0.csv")
    print(f" - {INDEXES_PATH}/graph_evidence_edges_v0.csv\n")
    
    display(GRAPH_EVIDENCE_NODES_V0.head(3))
    display(GRAPH_EVIDENCE_EDGES_V0.head(3))
    
    CHANGED_EVIDENCE_GRAPHS = True

else:
    print("Phase P4 skipped. No inputs changed.")

## Phase P5: Rebuild Structural Semantic Graphs

In [ ]:
# Phase P5: Rebuild Structural Semantic Graphs
LAST_PHASE_RUN = "P5"
CHANGED_SEMANTIC_GRAPHS = False

if (
    FORCE_REBUILD_SEMANTIC
    or CHANGED_ENTITIES
    or CHANGED_ALIASES
    or CHANGED_PREDICATES
    or CHANGED_RELATIONSHIPS
    or CHANGED_EVIDENCE_GRAPHS
):
    print("Phase P5 will run:")
    print(f"{'  Force rebuild semantic:':26} {FORCE_REBUILD_SEMANTIC}")
    print(f"{'  Changed entities:':26} {CHANGED_ENTITIES}")
    print(f"{'  Changed aliases:':26} {CHANGED_ALIASES}")
    print(f"{'  Changed predicates:':26} {CHANGED_PREDICATES}")
    print(f"{'  Changed relationsihps:':26} {CHANGED_RELATIONSHIPS}")
    print(f"{'  Changed evidence graphs:':26} {CHANGED_EVIDENCE_GRAPHS}")


    # ----------------------
    # Build nodes
    # ----------------------
    GRAPH_SEMANTIC_NODES_V0 = (
        pd.concat(
            [
                DF_RELATIONSHIP_SEMANTICS.loc[lambda df: df["include"], "subject_id"],
                DF_RELATIONSHIP_SEMANTICS.loc[lambda df: df["include"], "object_id"],
            ],
            ignore_index=True,
        )
        .drop_duplicates()
        .to_frame(name="node_id")
        .assign(
            node_type=lambda df: df["node_id"].str.split("_", n=1).str[0]
        )
        .merge(
            DF_VOCAB_ENTITIES[["entity_id", "canonical"]],
            left_on="node_id",
            right_on="entity_id",
            how="left",
        )
        .drop(columns="entity_id")
        .rename(columns={"canonical": "label"})
        .sort_values(["node_type", "node_id"], ascending=[True, True])
        .reset_index(drop=True)
    )

    # output
    print(f"\nSemantic nodes built: {len(GRAPH_SEMANTIC_NODES_V0)}")

    print("\nCounts by node_type:")
    display(GRAPH_SEMANTIC_NODES_V0["node_type"].value_counts().to_frame("count"))
    
    print("\nSample semantic nodes:")
    display(GRAPH_SEMANTIC_NODES_V0.sample(min(5, len(GRAPH_EVIDENCE_NODES_V0)), random_state=7))


    # ----------------------
    # Build edges
    # ----------------------
    GRAPH_SEMANTIC_EDGES_V0 = (
        DF_RELATIONSHIP_SEMANTICS
        .loc[lambda df: df["include"]]
        [
            [
                "subject_id",
                "predicate",
                "object_id",
                "subject_type",
                "object_type",
                "pair_type",
                "symmetric",
                "relationship_class",
            ]
        ]
        .sort_values(
            ["predicate", "subject_id", "object_id"],
            ascending=[True, True, True],
        )
    )

    # Output
    print(f"Semantic edges built: {len(GRAPH_SEMANTIC_EDGES_V0)}")
    print("Semantic edge counts by predicate:\n")
    
    display(
        GRAPH_SEMANTIC_EDGES_V0
            .groupby("predicate", as_index=False)
            .size()
            .rename(columns={"size": "edge_count"})
            .sort_values("edge_count", ascending=False)
            .reset_index(drop=True)
    )
    
    display(GRAPH_SEMANTIC_EDGES_V0.sample(3))
    
    # ----------------------
    # Write indexes out to files
    # ----------------------
    GRAPH_SEMANTIC_NODES_V0.to_csv(INDEXES_PATH / "graph_semantic_nodes_v0.csv", index=False, encoding="utf-8")
    GRAPH_SEMANTIC_EDGES_V0.to_csv(INDEXES_PATH / "graph_semantic_edges_v0.csv", index=False, encoding="utf-8")
    
    print(f" - {INDEXES_PATH}/graph_symantic_nodes_v0.csv")
    print(f" - {INDEXES_PATH}/graph_symantic_edges_v0.csv\n")
    
    display(GRAPH_SEMANTIC_NODES_V0.head(3))
    display(GRAPH_SEMANTIC_EDGES_V0.head(3))

    CHANGED_SEMANTIC_GRAPHS = True
# ----------------------
# resume overall structure here
# ----------------------
else:
    print("Phase P5 skipped. No inputs changed.")